In [1]:
from moviepy.editor import VideoFileClip, CompositeVideoClip, TextClip, ColorClip
from moviepy.video.tools.subtitles import SubtitlesClip
import moviepy.config as mpconfig
from moviepy.editor import VideoFileClip
import json
from pathlib import Path
import numpy as np
import pandas as pd
import os

In [2]:
def ensure_imagemagick():
    """Ensure ImageMagick is properly configured"""
    if os.name == 'nt':  # Windows
        # Try to find ImageMagick installation
        possible_paths = [
            r"C:\Program Files\ImageMagick-7.1.1-Q16-HDRI\magick.exe",
            r"C:\Program Files\ImageMagick-7.1.1-Q16\magick.exe",
            r"C:\Program Files (x86)\ImageMagick-7.1.1-Q16-HDRI\magick.exe",
            r"C:\Program Files (x86)\ImageMagick-7.1.1-Q16\magick.exe",
        ]
        
        imagemagick_path = None
        for path in possible_paths:
            if os.path.exists(path):
                imagemagick_path = path
                break
        
        if imagemagick_path:
            mpconfig.change_settings({"IMAGEMAGICK_BINARY": imagemagick_path})
            print(f"Using ImageMagick from: {imagemagick_path}")
        else:
            print("Warning: ImageMagick not found in common locations")
            mpconfig.change_settings({"IMAGEMAGICK_BINARY": "magick"})

# Call this function before any MoviePy operations


In [4]:
import pandas as pd
import numpy as np


In [5]:
def rolling_subtitle_data(words_and_times, max_words=5):
    """
    Build a list of ((start, end), text) tuples suitable for MoviePy's SubtitlesClip,
    using a rolling window of exactly max_words words, resetting after each chunk.
    The newest word will be in UPPERCASE.
    
    :param words_and_times: List of (word, start_time, end_time) for each word.
    :param max_words: Maximum number of words in the rolling window.
    :return: A list of ((start, end), text) intervals for MoviePy.
    """
    subtitles = []
    for i, (word, start_time, end_time) in enumerate(words_and_times):
        # Determine the time when this subtitle should stop
        if i + 1 < len(words_and_times):
            _, next_start, _ = words_and_times[i + 1]
            display_end = next_start
        else:
            display_end = end_time

        # Calculate which chunk this word belongs to and its position within the chunk
        chunk_number = i // max_words
        position_in_chunk = i % max_words
        
        # Get the start index for this chunk
        chunk_start = chunk_number * max_words
        
        # Get words for this chunk (up to the current word)
        visible_words = [w for (w, _, _) in words_and_times[chunk_start:chunk_start + position_in_chunk + 1]]
        
        # "Highlight" the newest word in uppercase
        visible_words[-1] = visible_words[-1].upper()
        
        # Join them all into a single subtitle line
        subtitle_text = " ".join(visible_words)
        
        # Append to our list in MoviePy's expected format
        subtitles.append(((start_time, display_end), subtitle_text))

    return subtitles

def make_rolling_subtitled_video(video_path, words_and_times, output_path, max_words=5):
    """
    Loads the video at `video_path`, generates rolling/karaoke-style subtitles
    from the `words_and_times` data, and writes out to `output_path`.
    """
    # 1) Generate the rolling-subtitle intervals for MoviePy
    sub_data = rolling_subtitle_data(words_and_times, max_words=max_words)

    # 2) Define how each subtitle line is rendered
    def subtitle_generator(txt):
        """
        A function that returns a TextClip for the given subtitle text `txt`.
        This function is called internally by SubtitlesClip for each subtitle segment.
        """
        # Create two TextClips - one for the normal text and one for the highlighted word
        words = txt.split()
        if not words:
            return TextClip("", fontsize=50, color='white')
            
        # Create clips for each word
        word_clips = []
        for i, word in enumerate(words):
            # Last word is highlighted in green and uppercase
            if i == len(words) - 1:
                clip = TextClip(
                    str(word),  # Convert to string explicitly
                    fontsize=50,
                    color='green',
                    font='Arial-Bold',
                    stroke_color='black',
                    stroke_width=2
                )
            else:
                clip = TextClip(
                    str(word),  # Convert to string explicitly
                    fontsize=50,
                    color='white',
                    font='Arial-Bold',
                    stroke_color='black',
                    stroke_width=2
                )
            word_clips.append(clip)
        
        # Calculate total width needed
        total_width = sum(clip.w for clip in word_clips) + (len(word_clips) - 1) * 10  # 10 pixels spacing
        
        # Create a blank clip to hold all words
        final_clip = ColorClip(size=(total_width, word_clips[0].h), color=(0,0,0))
        final_clip = final_clip.set_opacity(0)  # Make it transparent
        
        # Position each word
        x_pos = 0
        for clip in word_clips:
            final_clip = CompositeVideoClip([
                final_clip,
                clip.set_position((x_pos, 0))
            ])
            x_pos += clip.w + 10  # Add 10 pixels spacing between words
            
        return final_clip

    # 3) Create a SubtitlesClip from our list of timed subtitles
    subtitles_clip = SubtitlesClip(sub_data, subtitle_generator)

    # 4) Load the original video
    video_clip = VideoFileClip(video_path)

    # 5) Composite the video and subtitles together
    final_clip = CompositeVideoClip([video_clip, subtitles_clip.set_position(('center', 'bottom'))])

    # 6) Write the final video file
    final_clip.write_videofile(output_path, codec='libx264', fps=video_clip.fps)

    # Close the clips
    final_clip.close()
    video_clip.close()

In [6]:
import whisperx


In [7]:
def combine_chars_to_words(csv_path):
    df = pd.read_csv(csv_path)
    chars = df['a'].tolist()
    chars = chars[1:]
    starts = df['b'].tolist()
    starts = starts[1:]
    ends = df['c'].tolist()
    ends = ends[1:]
    words = []
    words_starts = []
    words_ends = []
    word = ""
    start_list = []
    end_list = []
    n = len(chars)
    for i in  range(n):
        char = chars[i]
        start = starts[i]
        end = ends[i]
        if char =="—":
            char = " "
            
        if char == " ":
            words.append(word)
            start_time = start_list[0]
            end_time = end_list[-1]
            start_list = []
            end_list = []
            words_starts.append(start_time)
            words_ends.append(end_time)
            word = ""
        else:
            word +=char
            start_list.append(start)
            end_list.append(end)
        
        if i ==n-1:
            words.append(word)
            start_time = start_list[0]
            end_time = end_list[-1]
            
            words_starts.append(start_time)
            words_ends.append(end_time)
            word = ""
            start_list = []
            end_list = []
    

    df = pd.DataFrame({"words": words, "words_starts": words_starts, "words_ends": words_ends})
    new_csv_path = csv_path.replace(".csv", "_combined.csv")
    df.to_csv(new_csv_path, index=False)
    return words, words_starts, words_ends

In [8]:
def words_to_combined_csv(csv_list, slide_duration=0.1, fade_duration = 0.1, fps = 24):
    combined_words = []
    combined_words_starts = []
    combined_words_ends = []
    previous_end = 0
    total_duration_prev = 0
    total_duration = 0
    for i in range(len(csv_list)):
        if i == 0:  
            words, words_starts, words_ends = combine_chars_to_words(csv_list[i])
            combined_words.extend(words)
            corrected_words_starts = [start +previous_end +1*(slide_duration +fade_duration) for start in words_starts]
            corrected_words_ends = [end +previous_end +1*(slide_duration +fade_duration) for end in words_ends]
            combined_words_starts.extend(corrected_words_starts)
            combined_words_ends.extend(corrected_words_ends)
            previous_end = corrected_words_ends[-1]
            total_duration_prev += words_ends[-1]
            total_duration += words_ends[-1] + 2*(slide_duration +fade_duration)
        else:
            words, words_starts, words_ends = combine_chars_to_words(csv_list[i])
            combined_words.extend(words)
            corrected_words_starts = [start +previous_end +(1+i/(5*(len(csv_list)-1)))*(2*slide_duration +2*fade_duration) for start in words_starts]
            corrected_words_ends = [end +previous_end +(1+i/(5*(len(csv_list)-1)))*(2*slide_duration +2*fade_duration) for end in words_ends]
            combined_words_starts.extend(corrected_words_starts)
            combined_words_ends.extend(corrected_words_ends)
            previous_end = corrected_words_ends[-1]
            total_duration_prev += words_ends[-1]
            total_duration += words_ends[-1] + 2*(slide_duration +fade_duration)

    return combined_words, combined_words_starts, combined_words_ends, total_duration, total_duration_prev





In [9]:
csv_list = []
csv_list_path = "generated_stories\example\speech"
num_files = 0
for file in os.listdir(csv_list_path):
    if file.endswith(".csv"):
        num_files +=1
num_files = num_files//2
for i in range(num_files):
    csv_list.append(csv_list_path + f"\p{i+1}.csv")

words, words_starts, words_ends, total_duration, total_duration_prev = words_to_combined_csv(csv_list)
dataframe = pd.DataFrame({"words": words, "words_starts": words_starts, "words_ends": words_ends})
dataframe.to_csv("combined_srt_csv.csv", index=False)




[('Hello', 0.0, 1.0), ('world', 1.0, 2.0), ('this', 2.0, 3.0), ('is', 3.0, 4.0), ('a', 4.0, 5.0), ('test', 5.0, 6.5), ('subtitle', 6.5, 7.5), ('rolling', 7.5, 8.5), ('example', 8.5, 10.0)]


In [15]:
from moviepy.editor import *
import pandas as pd
import os
import math


def make_rolling_subtitled_video(input_video_path, srt_data, output_path,
                                 chunk_size=5, y_offset=200, font_size=60):
    """
    - chunk_size: how many words to load at once
    - y_offset: distance from bottom of the video for subtitle
    """
    video = VideoFileClip(input_video_path)
    W, H = video.size
    font = "Arial-Bold"
    spacing = 10

    clips = [video]
    n_chunks = math.ceil(len(srt_data) / chunk_size)

    for ci in range(n_chunks):
        chunk = srt_data[ci*chunk_size : (ci+1)*chunk_size]
        if not chunk:
            break

        # Chunk timing: from first word start to last word end
        t_start = chunk[0][1]
        t_end   = chunk[-1][2]
        duration = t_end - t_start

        # Build the full-chunk base text (grey)
        # Join words with spaces
        line_text = " ".join(word for word, _, _ in chunk)
        base_txt = TextClip(line_text, fontsize=font_size, color="gray", font=font)
        # center horizontally
        x = (W - base_txt.w) // 2
        y = H - y_offset
        base = (base_txt
                .set_start(t_start)
                .set_duration(duration)
                .set_position((x, y)))
        clips.append(base)

        # Now overlay each word highlighted at its time
        # We need the x‐offsets of each word in the line
        # Simplest: measure each word's width to compute its x position
        x_cursor = x
        for word, w_start, w_end in chunk:
            txt = TextClip(word, fontsize=font_size, color="yellow", font=font)
            highlight = (txt
                         .set_start(w_start)
                         .set_duration(w_end - w_start)
                         .set_position((x_cursor, y)))
            clips.append(highlight)
            x_cursor += txt.w + spacing

    final = CompositeVideoClip(clips).set_duration(video.duration)
    final.write_videofile(output_path, fps=video.fps, codec="libx264")



if __name__ == "__main__":

    ensure_imagemagick()

    # Load the subtitle CSV — assumes format: word,start,end
    # data = pd.read_csv('combined_srt_csv.csv')
    data = [
        ("Hello",   0.0,  1.0),
        ("world",   1.0,  2.0),
        ("this",    2.0,  3.0),
        ("is",      3.0,  4.0),
        ("a",       4.0,  5.0),
        ("test",    5.0,  6.5),
        ("subtitle",6.5,  7.5),
        ("rolling", 7.5,  8.5),
        ("example", 8.5, 10.0),
    
    ]
    srt_data = data

    # Paths
    input_video = "generated_stories\\final_output\\final_video.mp4"
    output_video = "check1rolling_subs.mp4"

    # Generate video
    make_rolling_subtitled_video(input_video, srt_data, output_video, chunk_size=5)

    print("Done! Check:", output_video)


Using ImageMagick from: C:\Program Files\ImageMagick-7.1.1-Q16-HDRI\magick.exe
Moviepy - Building video check1rolling_subs.mp4.
MoviePy - Writing audio in check1rolling_subsTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video check1rolling_subs.mp4



Moviepy - Done !
Moviepy - video ready check1rolling_subs.mp4
Done! Check: check1rolling_subs.mp4


In [ ]:
from moviepy.editor import *
import pandas as pd
import os
import math


# base_color      = (179, 161, 201)   # "#B3A1C9"
# highlight_color = (94, 75, 139)     # "#5E4B8B"
# box_color       = (232, 226, 240)   # "#E8E2F0"



def make_rolling_subtitled_video(input_path, srt_data, output_path,
                                 chunk_size=5, y_offset=200, font_size=60,
                                 box_padding=10):
    def hex2rgb(h):
        h = h.lstrip('#')
        return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    video = VideoFileClip(input_path)
    W, H = video.size
    base_color      = "#7DF9FF"     # inactive words
    highlight_color = "#00FF41"     # current word
    box_hex       = "#DC143C"
    font = "Comic-Sans-MS-Bold-Italic"
    spacing = 10  # pixels between words
    box_color = hex2rgb(box_hex)

    clips = [video]
    n_chunks = math.ceil(len(srt_data) / chunk_size)

    for ci in range(n_chunks):
        chunk = srt_data[ci*chunk_size:(ci+1)*chunk_size]
        if not chunk:
            break

        # chunk timing
        t0 = chunk[0][1]
        t1 = chunk[-1][2]
        chunk_dur = t1 - t0

        # x start: center the chunk’s total width
        total_w = sum(TextClip(w, fontsize=font_size, font=font).w 
                      for w,_,_ in chunk) \
                  + spacing * (len(chunk)-1)
        x0 = (W - total_w) // 2
        y0 = H - y_offset

        x_cursor = x0

        for word, w_start, w_end in chunk:
            # 1) grey base word, visible for whole chunk
            base_txt = ( TextClip(word, fontsize=font_size, color=base_color, font=font)
                         .set_start(t0)
                         .set_duration(chunk_dur)
                         .set_position((x_cursor, y0)) )
            clips.append(base_txt)

            # 2) red box + green word, at exact same x_cursor
            highlighted  = TextClip(word, fontsize=font_size, color=highlight_color, font=font)
            boxed = ( highlighted 
                      .on_color(
                          size=(highlighted .w + box_padding,
                                highlighted .h + box_padding),
                          color=box_color,   # red
                          col_opacity=1
                      )
                      .set_start(w_start)
                      .set_duration(w_end - w_start)
                      .set_position((x_cursor - box_padding//2,
                                     y0 - box_padding//2)) )
            clips.append(boxed)

            x_cursor += highlighted .w + spacing

    final = CompositeVideoClip(clips).set_duration(video.duration)
    final.write_videofile(output_path, fps=video.fps, codec="libx264")

if __name__ == "__main__":

    ensure_imagemagick()

    # Load the subtitle CSV — assumes format: word,start,end
    # data = pd.read_csv('combined_srt_csv.csv')
    data = [
        ("Hello",   0.0,  1.0),
        ("world",   1.0,  2.0),
        ("this",    2.0,  3.0),
        ("is",      3.0,  4.0),
        ("a",       4.0,  5.0),
        ("test",    5.0,  6.5),
        ("subtitle",6.5,  7.5),
        ("rolling", 7.5,  8.5),
        ("example", 8.5, 10.0),
    
    ]
    srt_data = data

    # Paths
    input_video = "generated_stories\\final_output\\final_video.mp4"
    output_video = "check1rolling_subs.mp4"

    # Generate video
    make_rolling_subtitled_video(input_video, srt_data, output_video, chunk_size=5)

    print("Done! Check:", output_video)


Using ImageMagick from: C:\Program Files\ImageMagick-7.1.1-Q16-HDRI\magick.exe
Moviepy - Building video check1rolling_subs.mp4.
MoviePy - Writing audio in check1rolling_subsTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video check1rolling_subs.mp4



Moviepy - Done !
Moviepy - video ready check1rolling_subs.mp4
Done! Check: check1rolling_subs.mp4


In [16]:
if __name__ == "__main__":

    # Suppose you have a list of words, with their (start, end) times in seconds
    # In a real workflow, you'd extract or align these from your audio/dialogue

    # data = [
    #     ("Hello",   0.0,  1.0),
    #     ("world",   1.0,  2.0),
    #     ("this",    2.0,  3.0),
    #     ("is",      3.0,  4.0),
    #     ("a",       4.0,  5.0),
    #     ("test",    5.0,  6.5),
    #     ("subtitle",6.5,  7.5),
    #     ("rolling", 7.5,  8.5),
    #     ("example", 8.5, 10.0),
    
    # ]
    ensure_imagemagick()
    data = pd.read_csv('combined_srt_csv.csv')
    srt_data = list(zip(data.iloc[:, 0], data.iloc[:, 1], data.iloc[:, 2]))


    # Paths
    input_video = "generated_stories\\final_output\\final_video.mp4"             # Replace with your actual video file
    output_video = "check2rolling_subs.mp4"   # Name for the output
    ensure_imagemagick()
    # Generate the final video with rolling subs
    make_rolling_subtitled_video(input_video, srt_data, output_video, chunk_size=5)
    print("Done! Check:", output_video)

Using ImageMagick from: C:\Program Files\ImageMagick-7.1.1-Q16-HDRI\magick.exe
Using ImageMagick from: C:\Program Files\ImageMagick-7.1.1-Q16-HDRI\magick.exe
Moviepy - Building video check2rolling_subs.mp4.
MoviePy - Writing audio in check2rolling_subsTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video check2rolling_subs.mp4



Moviepy - Done !
Moviepy - video ready check2rolling_subs.mp4
Done! Check: check2rolling_subs.mp4


In [52]:
import requests
import base64

# ==== CONFIGURATION ====
API_KEY = "sk_69a9f580cd43d8d812c768e2d304f49ac81fe6fef2e49934"  # Replace with your ElevenLabs API key
VOICE_ID = "CoQByuTrT9gbKYx6QFL6"  # Replace with the voice ID you want to use
TEXT = "Hello there! This is a test for generating speech with timestamps."

# ==== API ENDPOINT ====
url = f"https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}/with-timestamps"

# ==== REQUEST HEADERS ====
headers = {
    "xi-api-key": API_KEY,
    "Content-Type": "application/json"
}

# ==== REQUEST PAYLOAD ====
data = {
    "text": TEXT,
    "model_id": "eleven_multilingual_v2",  # You can use another if needed
    "voice_settings": {
        "stability": 0.55,
        "similarity_boost": 0.75
    }
}

# ==== SEND REQUEST ====
response = requests.post(url, headers=headers, json=data)

# ==== HANDLE RESPONSE ====
if response.status_code == 200:
    result = response.json()

    # Decode and save audio
    audio_data = base64.b64decode(result["audio_base64"])
    with open("check_output_audio.wav", "wb") as f:
        f.write(audio_data)
    print("✅ Audio saved as output_audio.wav")

    # Print alignment info (timestamps)
    alignment = result.get("alignment", {})
    characters = alignment.get("characters", [])
    starts = alignment.get("character_start_times_seconds", [])
    ends = alignment.get("character_end_times_seconds", [])
    dtype = [('a', 'U10'), ('b', 'f4'), ('c', 'f4')]
    array = np.empty((1, 1,1), dtype=dtype)

    print("\n🕒 Character-level Timing:")
    for char, start, end in zip(characters, starts, ends):
        new_triplet = np.array([[[(char, start, end)]]], dtype=dtype)
        array = np.append(array, new_triplet, axis=0)
        print(f"'{char}' starts at {start:.2f}s, ends at {end:.2f}s")
        
    print(array)
else:
    print(f"❌ Error: {response.status_code}")
    print(response.text)


✅ Audio saved as output_audio.wav

🕒 Character-level Timing:
'H' starts at 0.00s, ends at 0.09s
'e' starts at 0.09s, ends at 0.17s
'l' starts at 0.17s, ends at 0.21s
'l' starts at 0.21s, ends at 0.26s
'o' starts at 0.26s, ends at 0.33s
' ' starts at 0.33s, ends at 0.37s
't' starts at 0.37s, ends at 0.41s
'h' starts at 0.41s, ends at 0.45s
'e' starts at 0.45s, ends at 0.53s
'r' starts at 0.53s, ends at 0.57s
'e' starts at 0.57s, ends at 0.70s
'!' starts at 0.70s, ends at 0.73s
' ' starts at 0.73s, ends at 0.94s
'T' starts at 0.94s, ends at 0.97s
'h' starts at 0.97s, ends at 1.04s
'i' starts at 1.04s, ends at 1.09s
's' starts at 1.09s, ends at 1.13s
' ' starts at 1.13s, ends at 1.17s
'i' starts at 1.17s, ends at 1.22s
's' starts at 1.22s, ends at 1.25s
' ' starts at 1.25s, ends at 1.30s
'a' starts at 1.30s, ends at 1.33s
' ' starts at 1.33s, ends at 1.39s
't' starts at 1.39s, ends at 1.44s
'e' starts at 1.44s, ends at 1.54s
's' starts at 1.54s, ends at 1.59s
't' starts at 1.59s, ends at 

In [44]:
array[1][0][0][2]

np.float32(0.093)

In [53]:
df = pd.DataFrame(array.reshape(-1))
df.to_csv("srt_test.csv", index=False)
print(df)

    a      b      c
0      0.000  0.000
1   H  0.000  0.093
2   e  0.093  0.174
3   l  0.174  0.209
4   l  0.209  0.255
.. ..    ...    ...
62  a  3.344  3.413
63  m  3.413  3.460
64  p  3.460  3.518
65  s  3.518  3.599
66  .  3.599  3.901

[67 rows x 3 columns]


In [30]:
# Define the data type for the structured array
dtype = [('a', 'U10'), ('b', 'f4'), ('c', 'f4')]

# Create an empty structured array with the specified shape
empty_array = np.empty((1, 1,1), dtype=dtype)

# Define the new triplet to add
new_triplet = np.array([[[('new', 3.0, 4.0)]]], dtype=dtype)
nd_triplet = np.array([[('new', 5.0, 6.0)]], dtype=dtype)

# Append the new triplet to the existing array
# Reshape the new_triplet to match the shape of the array
array = np.append(empty_array, new_triplet, axis=0)
print(array)
# Store the new triplet in the empty array

print(empty_array)

[[[('', 0., 0.)]]

 [[('new', 3., 4.)]]]
[[[('', 0., 0.)]]]


In [17]:
np.array([['new', 3.0, 4.0]])

array([['new', '3.0', '4.0']], dtype='<U32')